# PLS_Regression — walk-forward

**Lê:** `modeling_frame.csv` + `best_params_PLS_Regression.json`

**Grava:** `predictions_PLS_Regression.csv`, `metrics_PLS_Regression.json`, `feature_importance_PLS_Regression.csv`

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown
import pandas as pd

# sobe até achar analyses/grupo2
here = Path.cwd().resolve()
group = None
for c in [here, *here.parents]:
    if (c / 'config.py').exists() and (c / 'lib').is_dir():
        group = c
        break
if group is None:
    raise RuntimeError('Abra/rode este notebook de dentro de analyses/grupo2/...')
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib
from lib import features, eda_stl, models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print('grupo2 →', group)
print('outputs →', config.OUTPUT_DIR)


In [ ]:
model_df = pd.read_csv(require_file(config.DATA_DIR / 'modeling_frame.csv', '01_dados/03_feature_engineering.ipynb'),
                       parse_dates=[config.DATETIME_COL])
info = load_json(require_file(config.DATA_DIR / 'split_info.json', '01_dados/03_feature_engineering.ipynb'))
cut = int(info['cut_index'])
train = model_df.iloc[:cut].copy()
test = model_df.iloc[cut:].copy()
print('train', len(train), 'test', len(test))


In [ ]:
params = load_json(require_file(config.RESULT_DIR / 'best_params_PLS_Regression.json',
                               '02_modelos/PLS_Regression/01_otimizacao_hiperparametros.ipynb'))
print('params:', params)
print('Walk-forward PLS_Regression (pode demorar)...')
res = models.walkforward_ml(model_df, cut, 'PLS_Regression', params)

res.feature_importance = models.permutation_importance_pls(train, params)
models.save_result(res, config.RESULT_DIR)
print('MAE =', res.mae)
print('Ljung-Box p =', res.ljung_box_pvalue)
display(res.predictions.tail())
